# Исследование условного диапазона локализации

Отчёт читает только сохранённые результаты 120 заранее зафиксированных аудиозапусков. Он не запускает симуляцию, не подбирает параметры GCC, SRP или трекера и не использует истину в online-оценивателях. Три реализации шума дают разведочную оценку и не характеризуют редкие отказы. Значения SNR относятся к синтетической модели и не являются SPL или дальностью обнаружения реального БПЛА.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'results' / 'localization_range_study').exists():
    ROOT = ROOT.parent
RESULTS = Path(os.environ.get('RANGE_STUDY_RESULTS', ROOT / 'results' / 'localization_range_study')).resolve()
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(exist_ok=True)
run = pd.read_csv(RESULTS / 'run_summary.csv')
station = pd.read_csv(RESULTS / 'station_summary.csv')
groups = pd.read_csv(RESULTS / 'group_summary.csv')
boundaries = pd.read_csv(RESULTS / 'range_boundaries.csv')
summary = json.loads((RESULTS / 'study_summary.json').read_text())
assert summary['audio_run_count'] == 120 and len(run) == 240
assert len(station) == 720
summary

## Критерий

Точка проходит основной критерий при подтверждении не позднее 3 с, доступности после первых 3 с не ниже 90% и условном P95 ошибки не выше 5 м. Для фиксированного фона должны пройти все три реализации шума. Пороги 2 и 10 м показаны как анализ чувствительности.

In [ ]:
boundaries.sort_values(['experiment','trajectory','snr_db','estimator_variant','p95_threshold_m']).reset_index(drop=True)

In [ ]:
fixed_station = station[station.experiment == 'fixed_background'].copy()
snr_curve = (fixed_station.groupby(['trajectory','snr_db','distance_m','station_id'], as_index=False)
             .actual_station_snr_db.mean())
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (trajectory, data) in zip(axes, snr_curve.groupby('trajectory')):
    for (snr_ref, station_id), values in data.groupby(['snr_db','station_id']):
        ax.plot(values.distance_m, values.actual_station_snr_db, marker='o',
                label=f'{station_id}, ref {snr_ref:g} dB')
    ax.set(title=trajectory, xlabel='Начальное расстояние до центра, м', xscale='log')
    ax.grid(True, which='both', alpha=.3)
axes[0].set_ylabel('Фактический SNR станции, дБ')
axes[1].legend(fontsize=8, ncol=2)
fig.suptitle('Фиксированный фон: фактический SNR')
fig.tight_layout()
fig.savefig(FIGURES / 'actual_station_snr.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
fixed = run[run.experiment == 'fixed_background'].copy()
agg = (fixed.groupby(['trajectory','estimator_variant','snr_db','distance_m'])
       .position_p95_m_conditional.agg(['min','median','max']).reset_index())
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)
for ax, ((trajectory, method), data) in zip(axes.flat, agg.groupby(['trajectory','estimator_variant'])):
    for snr_ref, values in data.groupby('snr_db'):
        ax.plot(values.distance_m, values['median'], marker='o', label=f'{snr_ref:g} dB')
        ax.fill_between(values.distance_m, values['min'], values['max'], alpha=.18)
    ax.axhline(5, color='black', linestyle='--', linewidth=1, label='критерий 5 м')
    ax.set(title=f'{trajectory} / {method}', xscale='log', yscale='log',
           xlabel='Начальное расстояние, м', ylabel='Условный P95, м')
    ax.grid(True, which='both', alpha=.3)
    ax.legend(fontsize=8)
fig.suptitle('Ошибка координат при фиксированном фоне; полоса = min–max трёх шумов')
fig.tight_layout()
fig.savefig(FIGURES / 'fixed_background_position_p95.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
availability = (fixed.groupby(['trajectory','estimator_variant','snr_db','distance_m'])
                .availability_after_3s_fraction.agg(['min','median','max']).reset_index())
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
for ax, ((trajectory, method), data) in zip(axes.flat, availability.groupby(['trajectory','estimator_variant'])):
    for snr_ref, values in data.groupby('snr_db'):
        ax.plot(values.distance_m, values['median'], marker='o', label=f'{snr_ref:g} dB')
        ax.fill_between(values.distance_m, values['min'], values['max'], alpha=.18)
    ax.axhline(.9, color='black', linestyle='--', linewidth=1)
    ax.set(title=f'{trajectory} / {method}', xscale='log', ylim=(-.03, 1.03),
           xlabel='Начальное расстояние, м', ylabel='Доступность после 3 с')
    ax.grid(True, which='both', alpha=.3)
    ax.legend(fontsize=8)
fig.suptitle('Доступность сопровождения; полоса = min–max трёх шумов')
fig.tight_layout()
fig.savefig(FIGURES / 'fixed_background_availability.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
geometry = run[run.experiment == 'geometry_control'].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (trajectory, data) in zip(axes, geometry.groupby('trajectory')):
    for method, values in data.groupby('estimator_variant'):
        ax.plot(values.distance_m, values.position_p95_m_conditional, marker='o', label=method)
    ax.axhline(5, color='black', linestyle='--', linewidth=1)
    ax.set(title=trajectory, xlabel='Начальное расстояние, м', xscale='log', yscale='log')
    ax.grid(True, which='both', alpha=.3)
axes[0].set_ylabel('Условный P95, м')
axes[1].legend(fontsize=8)
fig.suptitle('Контроль геометрии при принятом SNR +10 дБ')
fig.tight_layout()
fig.savefig(FIGURES / 'geometry_control_position_p95.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
failure_table = (run.groupby(['experiment','trajectory','snr_db','distance_m','estimator_variant'], as_index=False)
                 .agg(first_confirmation_s=('first_confirmation_time_s_relative','max'),
                      availability_min=('availability_after_3s_fraction','min'),
                      losses=('track_loss_count','sum'),
                      recoveries=('recovery_count','sum'),
                      accepted=('accepted_update_count','sum'),
                      rejected=('rejected_update_count','sum')))
(failure_table[(failure_table.losses > 0) | (failure_table.availability_min < .9) |
              (failure_table.first_confirmation_s > 3) | failure_table.first_confirmation_s.isna()]
 .reset_index(drop=True))

## Интерпретация

Таблица `range_boundaries.csv` является основным выводом. Она сообщает только наибольшую прошедшую точку из заданной сетки и отдельно отмечает немонотонность. Прохождение 1000 м формулируется как «граница не найдена в исследованном диапазоне». Экстраполяция за сетку не выполняется. Ошибки RMSE/P95 условны по наличию действительной публикации; пропуски остаются в доступности.